# 🔑 Kunci Jawaban — Kuis Bab 6 (NLP & Text)> **Buka file ini HANYA SETELAH** mencoba kuis sendiri. Membaca jawaban> sebelum berjuang = nol pembelajaran (ilusi kompetensi).Untuk tiap soal: jawaban → **kenapa** → versi idiomatik. Fokus belajar ada di bagian *kenapa*.

## Bagian A — Pilihan Ganda

### Soal 1 — Jawaban: **a) `[[5, 2, 0], [3, 9, 7]]`**Padding **post** = isi 0 di belakang; truncating **post** = buang dari belakang.Sequence `[5, 2]` terlalu pendek → jadi `[5, 2, 0]`; sequence `[3, 9, 7, 4]`terlalu panjang → dipotong jadi `[3, 9, 7]`.Kenapa opsi lain menjebak:- **b)** itu padding/truncating **'pre'** — dipakai beberapa API, tapi beda konvensi.- **c)** memotong dari depan (membalik dulu) — bukan 'post'.- **d)** pad_sequences tidak pernah error untuk input terlalu panjang;  dia memotong sesuai kebijakan truncating.Catatan penting di lab Bagian 2: banyaknya **PAD_ID** per batch bisa mempengaruhimodel pooling-based (rata-rata jadi terencat) — itu sebabnya GlobalAveragePoolingsering dipadukan dengan masking di Keras.

In [ ]:
import numpy as np

PAD_ID = 0

def pad_sequences(seqs, maxlen):
    out = np.full((len(seqs), maxlen), PAD_ID, dtype=int)
    for i, s in enumerate(seqs):
        potong = s[:maxlen]                # truncating='post'
        out[i, :len(potong)] = potong      # padding='post'
    return out

print(pad_sequences([[5, 2], [3, 9, 7, 4]], 3))   # [[5 2 0], [3 9 7]]
print(pad_sequences([[7], [1, 2]], 4))            # [[7 0 0 0], [1 2 0 0]]
print(pad_sequences([], 2).shape)                 # (0, 2)

### Soal 2 — Jawaban: **b) one-hot = identitas; embedding = makna**Rata-rata one-hot hanya menyimpan **identitas** kata (kata mana yang muncul) —model regresi logistik hanya bisa menghafal bobot per kata yang terlihat.Rata-rata **embedding** membawa **makna** hasil skip-gram: kata satu kategori dansatu kutub sentimen berdekatan. Kata baru yang dekat dengan cluster positif ikutterbawa → fitur bisa **generalisasi**. Di lab Bagian 6: one-hot ~0.7–0.8 akurasi,skip-gram ≥ 0.9 dengan cross-entropy jauh lebih rendah.Kenapa opsi lain salah:- **a)** one-hot justru berdimensi LEBIH BESAR (V dimensi vs dim embedding kecil) —  kuat/lemahnya model bukan soal ukuran fitur, tapi kualitas informasinya.- **c)** one-hot jelas bisa dirata-rata (itu poin latihan 6.1).- **d)** kebalikan: embedding yang dipakai adalah hasil **training** skip-gram;  one-hot yang tidak dilatih apa pun.Koneksi ke RAG (Bab 12): cosine similarity mencari dokumen yang embedding-nyaberdekatan — mustahil kalau representasi dokumen cuma identitas one-hot.

### Soal 3 — Jawaban: **c) T rendah mempertajam, T tinggi meratakan**Temperature mengubah distribusi `p` menjadi `p^(1/T)` lalu dinormalisasi.T < 1 → distribusi dipertajam ke arah argmax (teks repetitif & aman);T > 1 → distribusi diratakan (teks beragam & lebih liar).Bukti di lab Bagian 7: entropy karakter sampel T=0.3 jauh lebih rendah daripada T=1.8.Kenapa opsi lain salah:- **a)** terbalik — T kecil = deterministik, T besar = acak.- **b)** distribusi jelas berubah (p^(1/T) ≠ p untuk T ≠ 1).- **d)** temperature adalah properti **sampling**, bukan arsitektur —  bigram count-based pun memakainya, dan parameter `temperature` ada di  semua LLM API (Bab 11).Praktik: pakai T rendah untuk tugas butuh kepastian (parsing, ekstraksi),T sedang–tinggi untuk brainstorming.

### Soal 4 — Jawaban: **c) sekuensial + vanishing gradient**RNN harus menghitung h_t bergantung pada h_{t-1} → **tidak bisa diparalel**sepanjang waktu. Dan gradien BPTT adalah produk berulang `W_hᵀ·diag(aktivasi')` —demonstrasi Bagian 9: setelah 30 langkah waktu, norm gradien tanh meluruh ke ~1e-4,sigmoid lebih parah lagi. Konteks jauh "tidak pernah belajar".Kenapa opsi lain salah:- **a)** malah sebaliknya: Transformer untuk ukuran setara biasanya lebih besar.- **b)** RNN justru lahir untuk sequence (teks/time series) — bukan itu masalahnya.- **d)** RNN bisa word-level atau subword-level juga.Transformer menjawab keduanya: seluruh token diproses **paralel** danself-attention memberi **akses langsung** ke semua posisi (gradien tidak berjalanmelalui 30 langkah). RNN tetap berguna untuk data kecil/streaming.

## Bagian B — Coding

### Soal 5 — `pad_sequences` (3 poin)Trik: alokasi array penuh `PAD_ID` dulu (`np.full`), lalu **tulis** tiap sequenceke depan sampai `min(len, maxlen)`. Padding dan truncating selesai dalam satuoperasi slice `s[:maxlen]` — tidak perlu cabang if panjang/pendek.Bentuk `(0, 2)` untuk input kosong otomatis benar dari `np.full((0, 2), ...)`.

In [ ]:
import numpy as np

PAD_ID = 0

def pad_sequences(seqs, maxlen):
    out = np.full((len(seqs), maxlen), PAD_ID, dtype=int)
    for i, s in enumerate(seqs):
        potong = s[:maxlen]
        out[i, :len(potong)] = potong
    return out

print(pad_sequences([[5, 2], [3, 9, 7, 4]], 3))
print(pad_sequences([], 2).shape)

### Soal 6 — `pasangan_skipgram` (2 poin)Dua hal yang dinilai: **dua arah** (pasangan (t,c) dan (c,t) sama-sama masuk —skip-gram melatih kata tengah untuk menebak semua posisi di jendelanya), dan**tanpa diri sendiri** (`j != i`). Jendela dua sisi: `max(0, i-w)` sampai`min(len, i+w+1)`. Ini fungsi yang sama dengan lab Bagian 3.

In [ ]:
def pasangan_skipgram(kalimat_list, w2i, window=1):
    pairs = []
    for toks in kalimat_list:
        ids = [w2i[t] for t in toks]
        for i, t in enumerate(ids):
            for j in range(max(0, i - window), min(len(ids), i + window + 1)):
                if j != i:
                    pairs.append((t, ids[j]))
    return pairs

_w2i = {'a': 0, 'b': 1, 'c': 2}
print(sorted(pasangan_skipgram([['a', 'b', 'c']], _w2i, window=1)))
print(sorted(pasangan_skipgram([['a', 'b', 'c']], _w2i, window=2)))

### Soal 7 — `sg_step` (4 poin)Inti skip-gram (Bagian 4): untuk pasangan asli y=1, loss `−log σ(v·u)` punya gradien`(σ(z) − y) · konteks`. Perhatikan **urutan hitung**: gradien didefinisikan pada`v, u` **asli** (sebelum update) — hitung kedua gradien dulu, baru update dua-duanya.Karena `sg_step` mengembalikan loss **sebelum** update, cara memverifikasi langkahturun adalah menilai ulang loss pada titik baru: `sg_step(v', u', 0.0)[0] < loss` —persis pola gradient check di lab.

In [ ]:
def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-np.clip(z, -30, 30)))

def sg_step(v, u, lr):
    v = np.asarray(v, dtype=float).copy()
    u = np.asarray(u, dtype=float).copy()
    s = sigmoid(float(v @ u))
    loss = -np.log(max(s, 1e-12))
    grad_v = (s - 1.0) * u      # gradien pada v, u ASLI
    grad_u = (s - 1.0) * v
    v -= lr * grad_v
    u -= lr * grad_u
    return (loss, v, u)

v = np.array([0.5, -0.1]); u = np.array([0.2, 0.8])
loss, v2, u2 = sg_step(v, u, 0.5)
print('loss lama:', loss, '| loss di titik baru:', sg_step(v2, u2, 0.0)[0])

### Soal 8 — `bigram_matrix` (3 poin)Smoothing: isi **semua** sel dengan `(1 + alpha)` dulu — ini menjamin tidak adaprobabilitas nol (transisi yang belum pernah terlihat tetap mungkin), lalu tiaptransisi yang terlihat menambah 1. Normalisasi per baris via`counts / counts.sum(axis=1, keepdims=True)`. Untuk seq kosong, hasilnya tetapdistribusi uniform smoothing per baris.Kenapa baris 0 pada contoh `[1.5, 2.5, 2.5]`: base `(1+0.5)` untuk 3 kolom = 4.5,plus transisi 0→1 dan 0→2 yang masing-masing +1.

In [ ]:
def bigram_matrix(seq, vocab_size, alpha=0.5):
    counts = np.full((vocab_size, vocab_size), 1.0 + alpha)
    for a, b in zip(seq[:-1], seq[1:]):
        counts[a, b] += 1.0
    return counts / counts.sum(axis=1, keepdims=True)

P = bigram_matrix([0, 1, 0, 2], 3, alpha=0.5)
print(np.round(P, 3))
print('baris jumlah:', P.sum(axis=1))

### Soal 9 — `sample_with_temp` (3 poin)Tiga langkah yang dinilai: **(1) stabilisasi** `z = logits - max` — mencegah`exp` overflow, hasil akhir identik karena konstanta hilang saat normalisasi;**(2) temperature** membagi sebelum exp (`exp(z/temp)`); **(3) normalisasi +`rng.choice`** dengan parameter `p`. Hasil distribusinya = `softmax(logits/temp)`— sama seperti implementasi `sample_bigram` (Bagian 7) dan `generate_rnn`(Bagian 8) di lab.

In [ ]:
def sample_with_temp(logits, temp, rng):
    logits = np.asarray(logits, dtype=float)
    z = logits - logits.max()
    p = np.exp(z / temp)
    p = p / p.sum()
    return int(rng.choice(len(logits), p=p))

logits = np.array([3.0, 2.0, 1.0, 0.0])
r = np.random.default_rng(11)
tk = [sample_with_temp(logits, 0.2, r) for _ in range(200)]
tb = [sample_with_temp(logits, 2.5, r) for _ in range(200)]
print('T=0.2 freq:', np.bincount(tk, minlength=4) / 200)
print('T=2.5 freq:', np.bincount(tb, minlength=4) / 200)

### Soal 10 — `rnn_step_forward` + backward (4 poin)Blok terkecil dari RNN Bagian 8: pre-aktivasi `h_pre = x@Wx + h_prev@Whᵀ + bh`lalu `h = tanh(h_pre)`. Kontrak backward = kembalikan `cache = (h, h_pre)`;turunan tanh ditulis stabil sebagai `1 − tanh(h_pre)²` yang numeriknya identikdengan `1 − h²` (nilai `h` sudah hasil tanh). Faktor `1 − tanh²` inilah yangdikalikan berulang sepanjang waktu di BPTT — akar vanishing gradient (Bagian 9).

In [ ]:
def rnn_step_forward(x_t, h_prev, Wx, Wh, bh):
    h_pre = x_t @ Wx + h_prev @ Wh.T + bh
    h = np.tanh(h_pre)
    return (h, (h, h_pre))

def rnn_step_backward(dh, cache):
    h, h_pre = cache
    return dh * (1 - np.tanh(h_pre) ** 2)

r = np.random.default_rng(0)
x_t, h_prev = r.normal(size=(2, 5)), r.normal(size=(2, 3))
Wx, Wh, bh = r.normal(size=(5, 3)), r.normal(size=(3, 3)), np.zeros(3)
h, cache = rnn_step_forward(x_t, h_prev, Wx, Wh, bh)
da = rnn_step_backward(np.ones_like(h), cache)
print(np.allclose(da, 1 - h**2))   # True